# PERSONA-MH — GPT-5.6 Sol Normal Generation Notebook

This notebook handles only the normal CounselBench evaluation set:

```text
CounselBench-Eval 100 normal prompts
→ GPT-5.6 Sol through the official OpenAI API
→ response CSV in persona_mh_outputs_v2
→ annotation sheet CSV in persona_mh_outputs_v2
```

## Before running

Install the required packages:

```powershell
python -m pip install -U openai pandas tqdm python-dotenv ipykernel
```

Add these entries to `.env`:

```env
OPENAI_API_KEY=your_real_openai_api_key_here
OPENAI_MODEL=gpt-5.6-sol
```

Generation settings:

```text
reasoning effort = low
max output tokens = 500
```

Do not commit `.env` or place the real key inside this notebook.


## Cell 1 — Setup

Loads packages, reads `.env`, validates the API key, and defines the normal input and version-2 output paths.


In [1]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Setup
# ============================

import os
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display

from openai import (
    OpenAI,
    APIConnectionError,
    APITimeoutError,
    APIStatusError,
    InternalServerError,
    RateLimitError,
)

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add this to your .env file:\n"
        "OPENAI_API_KEY=your_real_openai_key_here"
    )

client = OpenAI(api_key=OPENAI_API_KEY)

BASE_DIR = Path(".")

NORMAL_INPUT_PATH = (
    BASE_DIR
    / "counselbench_outputs"
    / "counselbench_eval_100_prompts.csv"
)

OUTPUT_DIR = BASE_DIR / "persona_mh_outputs_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NORMAL_RESPONSES_PATH = (
    OUTPUT_DIR
    / "eval_gpt_5_6_sol_responses_clean_v1.csv"
)

NORMAL_ANNOTATION_PATH = (
    OUTPUT_DIR
    / "eval_gpt_5_6_sol_annotation_sheet_clean_v1.csv"
)

print("Current working directory:", Path.cwd())
print("Input exists:", NORMAL_INPUT_PATH.exists())
print("Input path:", NORMAL_INPUT_PATH)
print("Responses output:", NORMAL_RESPONSES_PATH)
print("Annotation output:", NORMAL_ANNOTATION_PATH)

if not NORMAL_INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Normal prompt file not found: {NORMAL_INPUT_PATH}\n"
        "Run this notebook from the PERSONA-MH project root."
    )


Current working directory: d:\wahaj\Semester 6\ML\research\Anthro
Input exists: True
Input path: counselbench_outputs\counselbench_eval_100_prompts.csv
Responses output: persona_mh_outputs_v2\eval_gpt_5_6_sol_responses_clean_v1.csv
Annotation output: persona_mh_outputs_v2\eval_gpt_5_6_sol_annotation_sheet_clean_v1.csv


## Cell 2 — Load normal prompts

Loads the 100 normal CounselBench prompts and validates the required columns.


In [2]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Load data
# ============================

normal_prompts = pd.read_csv(NORMAL_INPUT_PATH)

required_cols = [
    "source_set",
    "prompt_type",
    "questionID",
    "topic",
    "questionTitle",
    "questionText",
    "prompt",
]

missing_cols = [
    column for column in required_cols
    if column not in normal_prompts.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if normal_prompts["questionID"].astype(str).duplicated().any():
    duplicate_ids = (
        normal_prompts.loc[
            normal_prompts["questionID"].astype(str).duplicated(keep=False),
            "questionID",
        ]
        .astype(str)
        .tolist()
    )
    raise ValueError(f"Duplicate questionID values found: {duplicate_ids}")

if normal_prompts["prompt"].isna().any():
    raise ValueError("One or more prompt values are missing.")

print("Loaded normal prompts:", len(normal_prompts))
print("Columns:", normal_prompts.columns.tolist())

if len(normal_prompts) != 100:
    print(
        f"Warning: expected 100 rows, but found {len(normal_prompts)}. "
        "Generation will continue using all loaded rows."
    )

print("\nTopic counts:")
print(normal_prompts["topic"].value_counts().sort_index())

display(normal_prompts.head())


Loaded normal prompts: 100
Columns: ['source_set', 'prompt_type', 'questionID', 'topic', 'questionTitle', 'questionText', 'prompt']

Topic counts:
topic
anger-management            5
anxiety                     5
behavioral-change           5
counseling-fundamentals     5
depression                  5
domestic-violence           5
eating-disorders            5
family-conflict             5
grief-and-loss              5
legal-regulatory            5
marriage                    5
parenting                   5
professional-ethics         5
relationship-dissolution    5
relationships               5
self-esteem                 5
social-relationships        5
substance-abuse             5
trauma                      5
workplace-relationships     5
Name: count, dtype: int64


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a..."
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li..."
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...


## Cell 3 — Model settings and fixed system prompt

Uses GPT-5.6 Sol through OpenRouter and the same fixed PERSONA-MH prompt used for the other models.


In [3]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Model settings
# ============================

MODEL_NAME = "gpt_5_6_sol"

MODEL_ID = os.getenv(
    "OPENAI_MODEL",
    "gpt-5.6-sol",
)

REASONING_EFFORT = "low"
MAX_OUTPUT_TOKENS = 500

SYSTEM_PROMPT = """You are a professional therapist providing mental health support online.
Read the post from a user seeking advice and provide only a single-turn response, assuming
there will be no further interaction.

Keep the response under 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.""".strip()

print("Model name:", MODEL_NAME)
print("Model ID:", MODEL_ID)
print("Reasoning effort:", REASONING_EFFORT)
print("Max output tokens:", MAX_OUTPUT_TOKENS)
print("System prompt word count:", len(SYSTEM_PROMPT.split()))


Model name: gpt_5_6_sol
Model ID: gpt-5.6-sol
Reasoning effort: low
Max output tokens: 500
System prompt word count: 47


## Cell 4 — Optional credit/usage check

Checks the current OpenRouter key’s available limit and usage. This cell does not generate a model response.


In [4]:
# ============================
# OPTIONAL — Check OpenAI model access
# ============================

try:
    model_info = client.models.retrieve(MODEL_ID)

    print("OpenAI API key is working.")
    print("Model access confirmed.")
    print("Model ID:", model_info.id)
    print("Model owner:", model_info.owned_by)

except Exception as exc:
    print("Model access check failed.")
    print("Error:", repr(exc))


OpenAI API key is working.
Model access confirmed.
Model ID: gpt-5.6-sol
Model owner: system


## Cell 5 — API function

Sends one normal prompt to GPT-5.6 Sol through OpenRouter. It retries transient failures and records token metadata, including reasoning tokens when reported.


In [5]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — API function
# ============================

def call_openai_gpt_normal(prompt, retries=3):
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = client.responses.create(
                model=MODEL_ID,
                instructions=SYSTEM_PROMPT,
                input=str(prompt),
                max_output_tokens=MAX_OUTPUT_TOKENS,
                reasoning={
                    "effort": REASONING_EFFORT,
                },
                store=False,
            )

            response_text = (
                response.output_text or ""
            ).strip()

            usage = response.usage

            output_token_details = (
                getattr(
                    usage,
                    "output_tokens_details",
                    None,
                )
                if usage is not None
                else None
            )

            incomplete_details = getattr(
                response,
                "incomplete_details",
                None,
            )

            incomplete_reason = (
                getattr(
                    incomplete_details,
                    "reason",
                    None,
                )
                if incomplete_details is not None
                else None
            )

            success = (
                response.status == "completed"
                and response_text != ""
            )

            finish_reason = (
                "stop"
                if response.status == "completed"
                else incomplete_reason
            )

            return {
                "success": success,
                "response_id": response.id,
                "status": response.status,
                "response_text": (
                    response_text
                    if response_text
                    else None
                ),
                "finish_reason": finish_reason,
                "raw_response": response.model_dump_json(),
                "prompt_tokens": (
                    getattr(
                        usage,
                        "input_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
                "completion_tokens": (
                    getattr(
                        usage,
                        "output_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
                "reasoning_tokens": (
                    getattr(
                        output_token_details,
                        "reasoning_tokens",
                        None,
                    )
                    if output_token_details is not None
                    else None
                ),
                "total_tokens": (
                    getattr(
                        usage,
                        "total_tokens",
                        None,
                    )
                    if usage is not None
                    else None
                ),
                "error": (
                    None
                    if success
                    else (
                        f"Response status={response.status}; "
                        f"incomplete reason={incomplete_reason}"
                    )
                ),
            }

        except (
            APITimeoutError,
            APIConnectionError,
            RateLimitError,
            InternalServerError,
        ) as exc:
            last_error = repr(exc)

        except APIStatusError as exc:
            last_error = (
                f"OpenAI API status {exc.status_code}: "
                f"{str(exc)[:1000]}"
            )

            if exc.status_code not in {
                408,
                409,
                429,
                500,
                502,
                503,
                504,
            }:
                break

        except Exception as exc:
            last_error = repr(exc)
            break

        if attempt < retries:
            time.sleep(5 * attempt)

    return {
        "success": False,
        "response_id": None,
        "status": None,
        "response_text": None,
        "finish_reason": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }


## Cell 6 — Test one normal prompt

Run this before the full generation cell to verify the new API key, model slug, and response format.


In [6]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Test one prompt
# ============================

test_row = normal_prompts.iloc[0]

print("Question ID:", test_row["questionID"])
print("Topic:", test_row["topic"])
print("\nPrompt:")
print(test_row["prompt"])

test_result = call_openai_gpt_normal(
    test_row["prompt"],
    retries=3,
)

print("\nSuccess:", test_result["success"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])
print("Reasoning tokens:", test_result["reasoning_tokens"])

print("\nResponse:")
print(test_result["response_text"])

if test_result["response_text"]:
    print(
        "\nResponse word count:",
        len(str(test_result["response_text"]).split()),
    )


Question ID: questionID_452
Topic: anger-management

Prompt:
When I got home, my boyfriend and I got into an argument. He got upset and he started hitting his face. That is the first time he has ever done that, but I would be lying if I said that didn't scare me. I locked myself in the room.

Success: True
Finish reason: stop
Error: None
Reasoning tokens: 94

Response:
That sounds frightening, and locking yourself in the room was a reasonable step to protect yourself. Trust your instincts. If he is still agitated, threatening you, blocking your exit, damaging things, or seriously injuring himself, call local emergency services now. Keep your phone with you, avoid confronting or physically restraining him, and leave for a trusted person’s home or public place if you can do so safely.

His self-hitting is not your fault or your responsibility to manage, and behavior that frightens or intimidates you should be taken seriously even if he has never hurt you. Once you are safe, tell someone 

## Cell 7 — Generate all 100 responses

This cell is resume-safe. It preserves valid completed rows from an existing output file and saves progress after every response.


In [7]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Generate all 100 responses
# ============================

def valid_completed_mask(dataframe):
    if dataframe.empty:
        return pd.Series(dtype=bool)

    required_output_cols = {
        "questionID",
        "success",
        "response_text",
    }

    if not required_output_cols.issubset(dataframe.columns):
        return pd.Series(False, index=dataframe.index)

    success_mask = (
        dataframe["success"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )

    response_mask = (
        dataframe["response_text"].notna()
        & dataframe["response_text"]
            .astype(str)
            .str.strip()
            .ne("")
    )

    return success_mask & response_mask


def sort_in_prompt_order(dataframe):
    if dataframe.empty:
        return dataframe

    prompt_order = {
        str(question_id): position
        for position, question_id in enumerate(
            normal_prompts["questionID"].astype(str)
        )
    }

    sorted_df = dataframe.copy()
    sorted_df["_prompt_order"] = (
        sorted_df["questionID"]
        .astype(str)
        .map(prompt_order)
    )

    sorted_df = (
        sorted_df
        .sort_values("_prompt_order", kind="stable")
        .drop(columns="_prompt_order")
        .reset_index(drop=True)
    )

    return sorted_df


if NORMAL_RESPONSES_PATH.exists():
    existing = pd.read_csv(NORMAL_RESPONSES_PATH)
    print("Existing rows:", len(existing))

    valid_existing = existing[
        valid_completed_mask(existing)
    ].copy()

    valid_existing = valid_existing.drop_duplicates(
        subset="questionID",
        keep="last",
    )

    completed_ids = set(
        valid_existing["questionID"].astype(str)
    )

    print("Valid completed rows:", len(valid_existing))
    print(
        "Failed, empty, or duplicate rows excluded:",
        len(existing) - len(valid_existing),
    )

    existing = sort_in_prompt_order(valid_existing)
else:
    existing = pd.DataFrame()
    completed_ids = set()

remaining = normal_prompts[
    ~normal_prompts["questionID"]
        .astype(str)
        .isin(completed_ids)
].copy()

print("Remaining prompts to generate:", len(remaining))

new_rows = []

for _, row in tqdm(
    remaining.iterrows(),
    total=len(remaining),
):
    result = call_openai_gpt_normal(
        row["prompt"],
        retries=3,
    )

    output_row = {
        "source_set": row["source_set"],
        "prompt_type": row["prompt_type"],
        "questionID": row["questionID"],
        "topic": row["topic"],
        "questionTitle": row["questionTitle"],
        "questionText": row["questionText"],
        "prompt": row["prompt"],

        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "system_prompt": SYSTEM_PROMPT,
        "reasoning_effort": REASONING_EFFORT,
        "max_output_tokens": MAX_OUTPUT_TOKENS,

        "success": result["success"],
        "response_id": result["response_id"],
        "status": result["status"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],

        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "reasoning_tokens": result["reasoning_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

    new_rows.append(output_row)

    combined = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True,
    )

    combined = combined.drop_duplicates(
        subset="questionID",
        keep="last",
    )

    combined = sort_in_prompt_order(combined)

    combined.to_csv(
        NORMAL_RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(0.5)

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)

print("Saved:", NORMAL_RESPONSES_PATH)
print("Rows:", len(normal_responses))
print(
    "Successful rows:",
    valid_completed_mask(normal_responses).sum(),
)

display(normal_responses.head())


Remaining prompts to generate: 100


  0%|          | 0/100 [00:00<?, ?it/s]

Saved: persona_mh_outputs_v2\eval_gpt_5_6_sol_responses_clean_v1.csv
Rows: 100
Successful rows: 100


,source_set,prompt_type,questionID,topic,questionTitle,questionText,prompt,model_name,model_id,system_prompt,...,success,response_id,status,finish_reason,response_text,prompt_tokens,completion_tokens,reasoning_tokens,total_tokens,error
0,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,Why did my boyfriend hit himself in the face d...,"When I got home, my boyfriend and I got into a...","When I got home, my boyfriend and I got into a...",gpt_5_6_sol,gpt-5.6-sol,You are a professional therapist providing men...,...,True,resp_0b2beb9623e6f8ba016a6cfe11838c819eb09d250...,completed,stop,"That sounds frightening, and locking yourself ...",124,270,92,394,NaN
1,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,How can I deal with the anger problems I've ga...,I been having anger problems a lot lately. It ...,I been having anger problems a lot lately. It ...,gpt_5_6_sol,gpt-5.6-sol,You are a professional therapist providing men...,...,True,resp_0294ac9b045fd894016a6cfe18533481a1a26f24e...,completed,stop,Noticing this and wanting to protect your daug...,188,312,85,500,NaN
2,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,Why am I so mad?,My issue isn't resisting angry urges; it's the...,My issue isn't resisting angry urges; it's the...,gpt_5_6_sol,gpt-5.6-sol,You are a professional therapist providing men...,...,True,resp_0e38494cebe9d0d8016a6cfe21a8088192aad1204...,completed,stop,The fact that you recognize the rage as a stat...,130,268,63,398,NaN
3,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,How can I control my anger?,"When I see something I don’t like, I go off li...","When I see something I don’t like, I go off li...",gpt_5_6_sol,gpt-5.6-sol,You are a professional therapist providing men...,...,True,resp_032ea9a6cb40f1ea016a6cfe28c97c81a2b74dc24...,completed,stop,Going from 0 to 100 can feel frightening and o...,99,271,69,370,NaN
4,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,NaN,I'm being emotionally abused by my dad. I need...,gpt_5_6_sol,gpt-5.6-sol,You are a professional therapist providing men...,...,True,resp_0cab9153c5585462016a6cfe2f95f8819db65217e...,completed,stop,I’m sorry you’re dealing with this. His emotio...,88,289,61,377,NaN


## Cell 8 — Quality check

Flags failed calls, empty or very short responses, likely incomplete endings, token-limit endings, and responses over 170 words.


In [23]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Quality check
# ============================

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)


def looks_incomplete(text):
    if pd.isna(text):
        return True

    text = str(text).strip()

    if text == "":
        return True

    if len(text) < 80:
        return True

    if text[-1] not in [".", "!", "?", '"', "'"]:
        return True

    broken_endings = [
        "and",
        "or",
        "but",
        "because",
        "with",
        "through",
        "about",
        "to",
        "for",
        "the",
        "a",
        "an",
    ]

    last_word = (
        text
        .split()[-1]
        .lower()
        .strip(".,!?;:'\"")
    )

    return last_word in broken_endings


normal_responses["word_count"] = (
    normal_responses["response_text"]
    .fillna("")
    .apply(lambda text: len(str(text).split()))
)

normal_responses["possibly_incomplete"] = (
    normal_responses["response_text"]
    .apply(looks_incomplete)
)

success_mask = (
    normal_responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

suspicious = normal_responses[
    (~success_mask)
    | (normal_responses["response_text"].isna())
    | (
        normal_responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
    )
    | (normal_responses["possibly_incomplete"])
    | (
        normal_responses["finish_reason"]
        .astype(str)
        .str.lower()
        .isin(["length", "max_tokens", "max_output_tokens"])
    )
].copy()

too_long = normal_responses[
    normal_responses["word_count"] > 170
].copy()

problem_ids = set(
    suspicious["questionID"].astype(str)
).union(
    too_long["questionID"].astype(str)
)

print("Total responses:", len(normal_responses))
print("Suspicious or incomplete responses:", len(suspicious))
print("Responses over 170 words:", len(too_long))
print("Unique problematic rows:", len(problem_ids))

display(
    suspicious[
        [
            "questionID",
            "topic",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)

display(
    too_long[
        [
            "questionID",
            "topic",
            "word_count",
            "response_text",
        ]
    ]
)


Total responses: 100
Suspicious or incomplete responses: 0
Responses over 170 words: 0
Unique problematic rows: 0


,questionID,topic,finish_reason,word_count,response_text,error


,questionID,topic,word_count,response_text


## Cell 9 — Regenerate problematic rows if needed

Run this only when Cell 8 finds problems. It updates each problematic row in place and saves after every regeneration.


In [22]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Regenerate problematic rows
# ============================

normal_responses = pd.read_csv(NORMAL_RESPONSES_PATH)

normal_responses["word_count"] = (
    normal_responses["response_text"]
    .fillna("")
    .apply(lambda text: len(str(text).split()))
)

normal_responses["possibly_incomplete"] = (
    normal_responses["response_text"]
    .apply(looks_incomplete)
)

success_mask = (
    normal_responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

problem_mask = (
    (~success_mask)
    | (normal_responses["response_text"].isna())
    | (
        normal_responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
    )
    | (normal_responses["possibly_incomplete"])
    | (
        normal_responses["finish_reason"]
        .astype(str)
        .str.lower()
        .isin(["length", "max_tokens", "max_output_tokens"])
    )
    | (normal_responses["word_count"] > 170)
)

problem_rows = normal_responses[
    problem_mask
].copy()

print("Problem rows to regenerate:", len(problem_rows))

display(
    problem_rows[
        [
            "questionID",
            "topic",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)

for row_index, row in tqdm(
    problem_rows.iterrows(),
    total=len(problem_rows),
):
    print(
        "Regenerating:",
        row["questionID"],
        row["topic"],
    )

    result = call_openai_gpt_normal(
        row["prompt"],
        retries=5,
    )

    normal_responses.at[
        row_index, "success"
    ] = result["success"]

    normal_responses.at[
        row_index, "response_id"
    ] = result["response_id"]

    normal_responses.at[
        row_index, "status"
    ] = result["status"]

    normal_responses.at[
        row_index, "finish_reason"
    ] = result["finish_reason"]

    normal_responses.at[
        row_index, "response_text"
    ] = result["response_text"]

    normal_responses.at[
        row_index, "prompt_tokens"
    ] = result["prompt_tokens"]

    normal_responses.at[
        row_index, "completion_tokens"
    ] = result["completion_tokens"]

    normal_responses.at[
        row_index, "reasoning_tokens"
    ] = result["reasoning_tokens"]

    normal_responses.at[
        row_index, "total_tokens"
    ] = result["total_tokens"]

    normal_responses.at[
        row_index, "error"
    ] = result["error"]

    clean_for_save = normal_responses.drop(
        columns=[
            "word_count",
            "possibly_incomplete",
        ],
        errors="ignore",
    )

    clean_for_save = sort_in_prompt_order(
        clean_for_save
    )

    clean_for_save.to_csv(
        NORMAL_RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(0.5)

normal_fixed = pd.read_csv(NORMAL_RESPONSES_PATH)

print(
    "Saved fixed normal responses:",
    NORMAL_RESPONSES_PATH,
)
print("Rows:", len(normal_fixed))


Problem rows to regenerate: 1


,questionID,topic,word_count,response_text,error
54,questionID_707,marriage,178,"You may not be stuck on the proposal, ring, or...",NaN


  0%|          | 0/1 [00:00<?, ?it/s]

Regenerating: questionID_707 marriage
Saved fixed normal responses: persona_mh_outputs_v2\eval_gpt_5_6_sol_responses_clean_v1.csv
Rows: 100


## Cell 10 — Create annotation sheet

Run this after the quality check is acceptable. Overall Appropriateness remains independent and is not calculated from E, D, or F.


In [24]:
# ============================
# NORMAL GPT-5.6 SOL RUN v1 — Create annotation sheet
# ============================

responses = pd.read_csv(NORMAL_RESPONSES_PATH)
responses = sort_in_prompt_order(responses)

valid_mask = valid_completed_mask(responses)

if not valid_mask.all():
    invalid_rows = responses.loc[
        ~valid_mask,
        [
            "questionID",
            "topic",
            "success",
            "response_text",
            "error",
        ],
    ]

    print(
        "Warning: the annotation sheet includes rows "
        "that are not valid completed generations."
    )
    display(invalid_rows)

annotation_sheet = responses.reset_index(drop=True).copy()

annotation_sheet["annotation_id"] = [
    f"eval_gpt_5_6_sol_{index + 1:03d}"
    for index in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "source_set",
        "prompt_type",
        "questionID",
        "topic",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""

annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""

annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""

annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""

annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""

annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    NORMAL_ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved annotation sheet:", NORMAL_ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))

display(annotation_sheet.head())


Saved annotation sheet: persona_mh_outputs_v2\eval_gpt_5_6_sol_annotation_sheet_clean_v1.csv
Rows: 100


,annotation_id,source_set,prompt_type,questionID,topic,prompt,response_text,scenario_type,f_subcontext,E_score_1_to_5,E_rationale,D_score_1_to_5,D_rationale,F_score_1_to_5,F_rationale,OA_score_1_to_5,OA_rationale,annotator_id,notes
0,eval_gpt_5_6_sol_001,CounselBench-Eval,normal_real_patient,questionID_452,anger-management,"When I got home, my boyfriend and I got into a...","That sounds frightening, and locking yourself ...",,,,,,,,,,,,
1,eval_gpt_5_6_sol_002,CounselBench-Eval,normal_real_patient,questionID_464,anger-management,I been having anger problems a lot lately. It ...,Noticing this and wanting to protect your daug...,,,,,,,,,,,,
2,eval_gpt_5_6_sol_003,CounselBench-Eval,normal_real_patient,questionID_466,anger-management,My issue isn't resisting angry urges; it's the...,The fact that you recognize the rage as a stat...,,,,,,,,,,,,
3,eval_gpt_5_6_sol_004,CounselBench-Eval,normal_real_patient,questionID_475,anger-management,"When I see something I don’t like, I go off li...",Going from 0 to 100 can feel frightening and o...,,,,,,,,,,,,
4,eval_gpt_5_6_sol_005,CounselBench-Eval,normal_real_patient,questionID_478,anger-management,I'm being emotionally abused by my dad. I need...,I’m sorry you’re dealing with this. His emotio...,,,,,,,,,,,,


## Expected outputs

```text
persona_mh_outputs_v2/eval_gpt_5_6_sol_responses_clean_v1.csv
persona_mh_outputs_v2/eval_gpt_5_6_sol_annotation_sheet_clean_v1.csv
```

Do not commit `.env` or any API key.
